# 03 - Model Comparison

Forty-four models on identical terms: same split, same features, same sample
weights, same metrics. That uniformity is the point. A ranking across model
families means nothing if the candidates also differ in how they were fitted.

Two things get decided here. Which model families suit this problem, and how
performance should be measured at all, which turns out to be the harder
question.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bench as B
import models as M
from clean import apply_weight_scheme, expand_to_weighted_labels
from features import MODEL_FEATURES

pd.set_option("display.width", 150)

## Preparing the training data

Each cell contributes one row per intensity level that was actually reported
there, weighted by how many people chose it. A cell where twelve people said
MMI 4 and nine said MMI 5 contributes both rows rather than being flattened to
a single number.

Weights use the square root of the report count. Raw counts would be the
inverse variance optimal choice if reports were independent observations, but
five hundred people describing the same shaking at the same place are not five
hundred independent measurements, and count weighting would pull population
bias straight into the loss: cell influence would span 5 to 1386. Under square
root it spans 2.2 to 37.

In [ ]:
features = pd.read_csv("../data/processed/features.csv")

labels = apply_weight_scheme(
    expand_to_weighted_labels(features, feature_columns=MODEL_FEATURES + ["mmi_mean"]),
    scheme="sqrt",
)
labels["is_weekend"] = labels["is_weekend"].astype(int)

print(f"cells         {len(features):>8,}")
print(f"weighted rows {len(labels):>8,}")

## Splitting by earthquake, not by row

Magnitude, depth and time of day are properties of an event, so every cell of
one earthquake carries the same value. Splitting rows at random would put
cells from the same event on both sides, and a model could then recognise
events rather than learn how shaking attenuates. With 95 events those features
carry 95 independent observations between them, not 24,241.

The split is also stratified by magnitude. There is one earthquake above
magnitude 7.5 in the whole catalogue, so an unstratified split could leave a
model that has never seen strong shaking.

In [ ]:
train, test = M.split_by_event(labels, test_size=0.2)

for name, part in [("train", train), ("test", test)]:
    magnitudes = part.groupby("public_id")["magnitude"].first()
    print(f"  {name}: M{magnitudes.min():.1f} to M{magnitudes.max():.1f}, "
          f"largest three {sorted(magnitudes.round(1))[-3:]}")

## The reference points

Three, and every model is measured against them.

A constant prediction, which is the floor any real model must clear. The
classical attenuation equation, intensity as a linear function of magnitude
and log distance, which is the shape almost every published intensity
prediction equation takes. And a ceiling, described further down.

In [ ]:
baseline = M.AttenuationBaseline().fit(train, train["mmi"], sample_weight=train["weight"])
print(baseline.equation())

That fit is far shallower than published equations, which typically carry a
magnitude coefficient near 1.2 and a log distance coefficient near -1.5.

The relationships genuinely are weaker in felt report data. A report only
exists where somebody felt something, so at long range the data records the
upper tail of what was experienced rather than the average, which flattens the
apparent decay.

### The ceiling

There is a limit to how well any model predicting one value per cell can score
against individual reports, because people standing in the same square
kilometre disagree with each other.

In [ ]:
ceiling = M.report_level_metrics(test["mmi"], test["mmi_mean"], test["weight"])
floor = M.report_level_metrics(test["mmi"], baseline.predict(test), test["weight"])

print(f"predicting each cell's own observed mean : MAE {ceiling['mae']:.3f}")
print(f"the attenuation equation                 : MAE {floor['mae']:.3f}")
print(f"real headroom                            : {floor['mae'] - ceiling['mae']:.3f}")

Without that number the results below would look far worse than they are.

## The metric problem

Intensity prediction work usually quotes the percentage of predictions within
one MMI unit. Against continuous predictions it behaves badly.

In [ ]:
constants = pd.DataFrame([
    {"prediction": value,
     **{k: round(v, 3) for k, v in
        M.report_level_metrics(labels["mmi"], np.full(len(labels), value),
                               labels["weight"]).items() if k in ("within_1_mmi", "mae")}}
    for value in [3.5, 4.0, 4.2, 4.5, 5.0]
])
constants

Predicting exactly 4.0 scores 0.898. Predicting 4.2, a better estimate by
every other measure, scores 0.597.

The reason is that MMI 3, 4 and 5 are 90.6% of all reports, so a prediction of
exactly 4.0 sits within one unit of almost everything, while 4.2 falls more
than a unit from MMI 3. The metric rewards landing on an integer rather than
being close, so a constant can outscore a strictly better prediction.
The selected model in notebook 04 scores 0.739 this way against 0.876 for
a constant 4.0, despite lower error on every other measure.

Mean absolute error is not fooled: it ranks 4.0 above 4.2 by a sensible margin
rather than a cliff. Two changes follow. MAE becomes the primary score, and
within-1 is computed on the rounded prediction so both sides are integers.

### A better score for ordinal predictions

The Ranked Probability Score compares the predicted and observed cumulative
distributions. It respects the ordering, so predicting MMI 7 when the answer
is 8 costs a quarter of what predicting 3 costs. It is proper, so honest
uncertainty scores better than confident error. And it is continuous, so there
is no integer snapping to exploit. It is the standard score for ordinal
forecasts in meteorology and seismology.

In [ ]:
examples = [
    ("certain and correct", [0, 0, 1, 0, 0, 0], 5),
    ("certain, off by one", [0, 1, 0, 0, 0, 0], 5),
    ("certain, off by four", [0, 0, 0, 0, 0, 1], 4),
    ("uncertain but centred", [0, .25, .5, .25, 0, 0], 5),
    ("uniform guess", [1/6] * 6, 5),
]
pd.DataFrame([
    {"case": label,
     "RPS": round(M.ranked_probability_score(np.array([probs]), [truth], np.arange(3, 9)), 4)}
    for label, probs, truth in examples
])

## Running the bench

Ten model families rather than many variants of whatever wins, because a
family doing well for a reason is more informative than a leaderboard of
near-identical gradient boosters.

In [ ]:
results = B.run_bench(train, test, MODEL_FEATURES, verbose=False)
succeeded = results[~results["status"].str.startswith("failed")]

print(f"{len(results)} candidates, {len(succeeded)} completed")
succeeded.nsmallest(12, "cell_mae")[
    ["model", "family", "kind", "cell_mae", "rps", "report_within_1", "auc_mmi6plus"]
]

Tree and boosting families take the top places. Linear models cluster around
0.47. Discriminant analysis and naive Bayes do worse than predicting a
constant.

Against the references: the best model reaches 0.337 cell MAE where the
attenuation equation reaches 0.545 and a constant reaches 0.590, so roughly a
38% improvement over both.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ordered = succeeded.sort_values("cell_mae")
colours = ["firebrick" if family == "baseline" else "steelblue"
           for family in ordered["family"]]
ax.barh(ordered["model"], ordered["cell_mae"], color=colours)
ax.invert_yaxis()
ax.set_xlabel("Cell level mean absolute error (MMI)")
ax.set_title("Model comparison, reference points in red")
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

## Predicting the intensities that matter most

Aggregate error hides the levels with the highest human consequence. MMI 7 and
above is under 2% of reports, so a model that never predicts it can still
score well overall.

Judged by whether it names those levels, every model fails: the most likely
single level is essentially never 7 or 8 at that base rate. But naming is the
wrong question. What an operational system needs is a ranking of where
damaging shaking is most likely, and measured that way the models carry real
signal.

In [ ]:
succeeded.nlargest(8, "auc_mmi6plus")[
    ["model", "auc_mmi6plus", "auc_mmi7plus", "rps", "cell_mae", "recall_mmi7plus"]
]

ROC AUC around 0.80 for identifying MMI 6 and above, against per-class recall
of essentially zero for the same models. The models rank damaging cells
higher; they just cannot name them. Precision is limited by the base rate
rather than by ranking quality, which is the strongest argument in this
project for bringing in instrumental measurements alongside felt reports.

## Summary

- Boosted trees lead, beating both the classical attenuation equation and a
  constant by around 38%.
- Within-1 is unusable against continuous predictions and has been replaced by
  MAE and the Ranked Probability Score, with within-1 kept on rounded values
  for comparability.
- No model can name MMI 7 or 8, but the leaders rank damaging shaking well
  (AUC 0.80).

Accuracy alone does not settle which model to use. The next notebook applies
the constraint that matters more.

In [ ]:
results.to_csv("../data/processed/bench_results.csv", index=False)
print("saved data/processed/bench_results.csv")